# Codealong Notebook

Use this notebook as your "scratch pad" as you go through the course contents. Feel free to copy any example code and tweak it to get a better understanding of how it works!

Use the **+** button or `Insert` menu to add additional code cells as needed.

## Step 1

### Loading the Data with `pandas`

In [3]:
import requests

# Get the Wikipedia page for "2022" since OpenAI's models stop in 2021
params = {
    "action": "query", 
    "prop": "extracts",
    "exlimit": 1,
    "titles": "2022",
    "explaintext": 1,
    "formatversion": 2,
    "format": "json"
}
resp = requests.get("https://en.wikipedia.org/w/api.php", params=params)
response_dict = resp.json()
response_dict["query"]["pages"][0]["extract"].split("\n")

['2022 (MMXXII) was a common year starting on Saturday of the Gregorian calendar, the 2022nd year of the Common Era (CE) and Anno Domini (AD) designations, the 22nd  year of the 3rd millennium and the 21st century, and the  3rd   year of the 2020s decade.  ',
 'The year saw the removal of nearly all COVID-19 restrictions and the reopening of international borders in most countries, while the global rollout of COVID-19 vaccines continued. The global economic recovery from the pandemic continued, though many countries experienced an ongoing inflation surge; in response, many central banks raised their interest rates to landmark levels. The world population reached eight billion people in 2022. The year also witnessed numerous natural disasters, including two devastating Atlantic hurricanes (Fiona and Ian), and the most powerful volcano eruption of the century so far. The later part of the year also saw the first public release of ChatGPT by OpenAI starting an arms race in artificial inte

In [4]:
import pandas as pd

# Load page text into a dataframe
df = pd.DataFrame()
df["text"] = response_dict["query"]["pages"][0]["extract"].split("\n")
df

,text
0,2022 (MMXXII) was a common year starting on Sa...
1,The year saw the removal of nearly all COVID-1...
2,2022 was also dominated by wars and armed conf...
3,
4,
...,...
254,
255,== Nobel Prizes ==
256,
257,


In [5]:
from dateutil.parser import parse

# Clean up text to remove empty lines and headings
df = df[(df["text"].str.len() > 0) & (~df["text"].str.startswith("=="))]

# In some cases dates are used as headings instead of being part of the
# text sample; adjust so dated text samples start with dates
prefix = ""
for (i, row) in df.iterrows():
    # If the row already has " - ", it already has the needed date prefix
    if " – " not in row["text"]:
        try:
            # If the row's text is a date, set it as the new prefix
            parse(row["text"])
            prefix = row["text"]
        except:
            # If the row's text isn't a date, add the prefix
            row["text"] = prefix + " – " + row["text"]
df = df[df["text"].str.contains(" – ")].reset_index(drop=True)
df


,text
0,– 2022 (MMXXII) was a common year starting on...
1,– The year saw the removal of nearly all COVI...
2,– 2022 was also dominated by wars and armed c...
3,January 1 – The Regional Comprehensive Econom...
4,January 2 – Abdalla Hamdok resigns as Prime Mi...
...,...
180,December 21–December 26 – A major winter storm...
181,December 24 – 2022 Fijian general election: Th...
182,December 29 – Brazilian football legend Pelé d...
183,December 31 – Former Pope Benedict XVI dies at...


### Creating an Embeddings Index with `openai.Embedding`

In [6]:
import os
import openai

openai.api_key = os.getenv("OPENAI_API_KEY")

In [7]:
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"
batch_size = 100
embeddings = []
for i in range(0, len(df), batch_size):
    # Send text data to OpenAI model to get embeddings
    response = openai.Embedding.create(
        input=df.iloc[i:i+batch_size]["text"].tolist(),
        engine=EMBEDDING_MODEL_NAME
    )

    # Add embeddings to list
    embeddings.extend([data["embedding"] for data in response["data"]])

# Add embeddings list to dataframe
df["embeddings"] = embeddings
df

,text,embeddings
0,– 2022 (MMXXII) was a common year starting on...,"[9.040244185598567e-05, -0.017994549125432968,..."
1,– The year saw the removal of nearly all COVI...,"[-0.010697541758418083, -0.023004746064543724,..."
2,– 2022 was also dominated by wars and armed c...,"[-0.009615330025553703, -0.015318214893341064,..."
3,January 1 – The Regional Comprehensive Econom...,"[-0.0005404727999120951, -0.024158069863915443..."
4,January 2 – Abdalla Hamdok resigns as Prime Mi...,"[-0.014883670955896378, 0.0011910186149179935,..."
...,...,...
180,December 21–December 26 – A major winter storm...,"[-0.024850230664014816, -0.023943288251757622,..."
181,December 24 – 2022 Fijian general election: Th...,"[-0.011599518358707428, -0.00938243605196476, ..."
182,December 29 – Brazilian football legend Pelé d...,"[-0.007591542787849903, 0.004114208742976189, ..."
183,December 31 – Former Pope Benedict XVI dies at...,"[0.023569442331790924, 0.007640643510967493, -..."


In [8]:
df.to_csv("embeddings.csv")

## Step 2

### Finding Relevant Data with Cosine Similarity

In [9]:
import os
import openai

#openai.api_base = "https://openai.vocareum.com/v1"
openai.api_key = os.getenv("OPENAI_API_KEY")

In [10]:
import numpy as np
import pandas as pd

df = pd.read_csv("embeddings.csv", index_col=0)
df["embeddings"] = df["embeddings"].apply(eval).apply(np.array)
df

,text,embeddings
0,– 2022 (MMXXII) was a common year starting on...,"[9.040244185598567e-05, -0.017994549125432968,..."
1,– The year saw the removal of nearly all COVI...,"[-0.010697541758418083, -0.023004746064543724,..."
2,– 2022 was also dominated by wars and armed c...,"[-0.009615330025553703, -0.015318214893341064,..."
3,January 1 – The Regional Comprehensive Econom...,"[-0.0005404727999120951, -0.024158069863915443..."
4,January 2 – Abdalla Hamdok resigns as Prime Mi...,"[-0.014883670955896378, 0.0011910186149179935,..."
...,...,...
180,December 21–December 26 – A major winter storm...,"[-0.024850230664014816, -0.023943288251757622,..."
181,December 24 – 2022 Fijian general election: Th...,"[-0.011599518358707428, -0.00938243605196476, ..."
182,December 29 – Brazilian football legend Pelé d...,"[-0.007591542787849903, 0.004114208742976189, ..."
183,December 31 – Former Pope Benedict XVI dies at...,"[0.023569442331790924, 0.007640643510967493, -..."


In [11]:
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"
USER_QUESTION = """What were the estimated damages of the 2023 \
Turkey-Syria earthquake?"""

# Generate the embedding response
response = openai.Embedding.create(
    input=[USER_QUESTION],
    engine=EMBEDDING_MODEL_NAME
)

# Extract the embeddings from the response
question_embeddings = response["data"][0]["embedding"]

print(question_embeddings[:100])

[0.00553178321570158, -0.024947911500930786, 0.002177121117711067, -0.012254414148628712, -0.02144855074584484, 0.002601235406473279, -0.03387593477964401, -0.013079358264803886, 0.0024498847778886557, -0.014982052147388458, 0.0162061620503664, 0.044413935393095016, -0.010231969878077507, -0.013179149478673935, 0.01508849672973156, -0.005395401734858751, 0.012400775216519833, -0.013039441779255867, 0.008202873170375824, -0.005724714137613773, -0.007444456685334444, 0.011476038955152035, 0.012387469410896301, -0.009120956063270569, 0.015194941312074661, 0.03209299221634865, 0.008049859665334225, -0.0023667251225560904, -0.0008465657592751086, -0.012141316197812557, 0.005505172535777092, -0.006552984472364187, -0.023524217307567596, 0.015261468477547169, -0.03246554732322693, -0.0075309425592422485, 0.006865664850920439, -0.004883137531578541, 0.023231495171785355, -0.0078436229377985, 0.010112219490110874, 0.028553714975714684, 0.012959607876837254, -0.007697261869907379, -0.01757663488

In [12]:
from openai.embeddings_utils import distances_from_embeddings

# Create a list containing the distances from question_embeddings
distances = distances_from_embeddings(
    question_embeddings,
    df["embeddings"],
    distance_metric="cosine"
)

print(distances[:100])

[0.2824168564616253, 0.2454698524500013, 0.217292355804789, 0.2724040373350154, 0.2655723973046469, 0.27733637386525967, 0.2445255873776606, 0.2615334626832, 0.2536834208890082, 0.2951562893280997, 0.3073406710962445, 0.2304318030641459, 0.2997100941592409, 0.2328279341200662, 0.27088533985704955, 0.3056408004590401, 0.26163075001818903, 0.2485099558489976, 0.26747340233187256, 0.3016659987655075, 0.22633539429117622, 0.306719254828678, 0.25189938491376473, 0.267311748273781, 0.2497621652816029, 0.2613268033394516, 0.2704718474943849, 0.26808112689563834, 0.24446174432773493, 0.2668961540089487, 0.27541650072494184, 0.24543230846283404, 0.2803507677456979, 0.27184282236072255, 0.2587530073678649, 0.23475380491838127, 0.26166318069490235, 0.2510088564656421, 0.2670030888799504, 0.28580654682131756, 0.23194116973460233, 0.27297992778327274, 0.2417437023469442, 0.2476178525744106, 0.27393073416254876, 0.25284762936582306, 0.27073072360160444, 0.3009327863180373, 0.22883305185088065, 0.235

In [13]:
df["distances"] = distances
df.sort_values(by="distances", ascending=True, inplace=True)
df.head(5)

,text,embeddings,distances
93,June 22 – A 6.2 earthquake strikes the Durand ...,"[-0.0017065758584067225, -0.002665197942405939...",0.178517
171,November 21 – A 5.6 earthquake strikes near Ci...,"[0.004124542232602835, -0.004101862199604511, ...",0.180608
109,July 27 – A 7.0 earthquake strikes the island ...,"[0.002066155895590782, 0.003913976717740297, 0...",0.189808
126,September 5 – A 6.8 earthquake strikes Luding ...,"[0.012683872133493423, 0.016343697905540466, 0...",0.201568
119,August 17 – Turkey and Israel agree to restore...,"[-0.010702475905418396, -0.014352227561175823,...",0.216978


In [14]:
df.to_csv("distances.csv")

## Step 3

### Tokenizing with `tiktoken`

In [24]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")
len(tokenizer.encode("Answer the question based on the context"))

7

### Composing a Custom Text Prompt

In [23]:
import tiktoken

MAX_TOKEN_COUNT = 200

# Create a tokenizer that is designed to align with our embeddings
tokenizer = tiktoken.get_encoding("cl100k_base")

# Count the number of tokens in the prompt template and question
prompt_template = """
Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context: 

{}

---

Question: {}
Answer:"""

current_token_count = len(tokenizer.encode(prompt_template)) + \
                        len(tokenizer.encode(USER_QUESTION))

context = []
for text in df["text"].values:

    # Increase the counter based on the number of tokens in this row
    text_token_count = len(tokenizer.encode(text))
    current_token_count += text_token_count

    # Add the row of text to the list if we haven't exceeded the max
    if current_token_count <= MAX_TOKEN_COUNT:
        context.append(text)
    else:
        break

prompt = prompt_template.format("\n\n###\n\n".join(context), USER_QUESTION)
print(prompt)


Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context: 

June 22 – A 6.2 earthquake strikes the Durand Line between Afghanistan and Pakistan, killing at least 1,163 people.

###

November 21 – A 5.6 earthquake strikes near Cianjur in West Java, Indonesia, killing 635 people and injuring 7,700 more.

###

July 27 – A 7.0 earthquake strikes the island of Luzon in the Philippines killing 11 people and injuring over 600.

###

September 5 – A 6.8 earthquake strikes Luding County in Sichuan province, China, killing 117 and injuring 424.

###

August 17 – Turkey and Israel agree to restore full diplomatic relations after a period of tensions.

---

Question: What were the estimated damages of the 2023 Turkey-Syria earthquake?
Answer:


In [ ]:
import tiktoken

def create_prompt(question, df, max_token_count):
    """
    Given a question and a dataframe containing rows of text and their
    embeddings, return a text prompt to send to a Completion model
    """
    # Create a tokenizer that is designed to align with our embeddings
    tokenizer = tiktoken.get_encoding("cl100k_base")

    # Count the number of tokens in the prompt template and question
    prompt_template = """
Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context: 

{}

---

Question: {}
Answer:"""

    current_token_count = len(tokenizer.encode(prompt_template)) + \
                            len(tokenizer.encode(question))

    context = []
    for text in get_rows_sorted_by_relevance(question, df)["text"].values:

        # Increase the counter based on the number of tokens in this row
        text_token_count = len(tokenizer.encode(text))
        current_token_count += text_token_count

        # Add the row of text to the list if we haven't exceeded the max
        if current_token_count <= max_token_count:
            context.append(text)
        else:
            break

    return prompt_template.format("\n\n###\n\n".join(context), question)

## Step 4

### Getting a Custom Q&A Response with `openai.Completion`

In [25]:
answer = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    max_tokens=150
)["choices"][0]["text"].strip()
print(answer)

I don't know
